# QLoRA + Dr.ICL

Fine-tune a small model with QLoRA and evaluate it with retrieval-augmented
in-context learning.

**Runtime > Change runtime type > T4 GPU** before running: 4-bit quantization
requires CUDA.

In [ ]:
!git clone https://github.com/CMUZrz/FineTuning_to_beat_llama_3.git
%cd FineTuning_to_beat_llama_3
%pip install -q -e .

## 1. Authenticate

Set `HUGGINGFACE_HUB_TOKEN` in the Colab **Secrets** panel (key icon in the
sidebar) rather than pasting a token into a cell.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

os.environ['HUGGINGFACE_HUB_TOKEN'] = userdata.get('HUGGINGFACE_HUB_TOKEN')
login(token=os.environ['HUGGINGFACE_HUB_TOKEN'])
print('Logged in to Hugging Face Hub')

## 2. Fine-tune

Note the **split fingerprint** this prints - the evaluation step must report
the same one, or the two commands are not talking about the same held-out set.

In [ ]:
!qlora-train \n  --model_id EleutherAI/pythia-410m \n  --torch_dataset_url lavita/ChatDoctor-HealthCareMagic-100k \n  --max_samples 4000 \n  --max_steps 200 \n  --project colab \n  --no_wandb \n  --no_push

## 3. Evaluate

`--adapter_id` loads the trained LoRA weights on top of the base model.
Without it you are scoring the base model. `--use_icl` retrieves
demonstrations from the *train* split only; an overlap with the evaluation
split raises rather than quietly inflating the score.

In [ ]:
!qlora-eval \n  --model_id EleutherAI/pythia-410m \n  --adapter_id ./pythia-410m-colab \n  --test_dataset lavita/ChatDoctor-HealthCareMagic-100k \n  --max_samples 4000 \n  --limit 32 \n  --use_icl --icl_top_k 3 \n  --no_wandb

## 4. How much would the classic evaluation bugs have been worth?

Scores the same model four ways, isolating retrieval leakage and prompt echo.

In [ ]:
!python scripts/demo_leakage.py \n  --model_id EleutherAI/pythia-410m \n  --adapter_id ./pythia-410m-colab \n  --limit 16